# 01 — Silver | Dimensions: Teams, Venues, Players and Official Standings

**Project:** MLB Performance Intelligence Platform

Parses the latest reference payloads from Bronze into conformed dimensions (upserted with `MERGE`).

| Table | Key | Source | Notes |
|---|---|---|---|
| `silver.dim_team` | `team_id` | `/teams` (latest season per team) | League, division, abbreviation, home venue |
| `silver.dim_venue` | `venue_id` | `/venues?hydrate=location` | City, state, country, coordinates, elevation |
| `silver.dim_player` | `player_id` | `/people?hydrate=currentTeam` | Birth country and `is_dominican` flag, position, bats/throws, debut |
| `silver.official_standings` | `season, team_id` | `/standings` (regular season) | Used only to **reconcile** the W-L and runs we compute from games |

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
import schemas as S
for _m in (C, P, S):
    importlib.reload(_m)
from pyspark.sql import functions as F, Window

RUN_ID = P.new_run_id()

def explode_latest(endpoint, schema, array_field):
    """Latest payload per request -> one row per array element, newest extraction first."""
    return (P.latest_payloads(spark, endpoint)
            .select(F.explode(F.from_json("payload", schema)[array_field]).alias("x"), "extracted_at"))

def keep_latest(df, keys, order_cols):
    w = Window.partitionBy(*keys).orderBy(*[F.col(c).desc() for c in order_cols])
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

## dim_team

In [ ]:
with P.log_step(spark, RUN_ID, "silver", "dim_team") as step:
    teams = explode_latest("teams", S.TEAMS, "teams").select(
        F.col("x.id").alias("team_id"),
        F.col("x.name").alias("team_name"),
        F.col("x.abbreviation").alias("team_abbreviation"),
        F.col("x.clubName").alias("club_name"),
        F.col("x.locationName").alias("location_name"),
        F.col("x.shortName").alias("short_name"),
        F.col("x.league.id").alias("league_id"),
        F.col("x.league.name").alias("league_name"),
        F.when(F.col("x.league.id") == 103, "AL").when(F.col("x.league.id") == 104, "NL").alias("league_abbreviation"),
        F.col("x.division.id").alias("division_id"),
        F.col("x.division.name").alias("division_name"),
        F.col("x.venue.id").alias("home_venue_id"),
        F.col("x.venue.name").alias("home_venue_name"),
        F.col("x.firstYearOfPlay").cast("int").alias("first_year_of_play"),
        F.col("x.season").alias("latest_season"),
        "extracted_at",
    )
    dim_team = keep_latest(teams, ["team_id"], ["latest_season", "extracted_at"]) \
        .withColumn("updated_at", F.current_timestamp()).drop("extracted_at")
    step.records = P.merge_upsert(spark, dim_team, C.table("silver", "dim_team"), ["team_id"],
        comment="Silver: MLB teams (current attributes from the latest season extracted)")

## dim_venue

In [ ]:
with P.log_step(spark, RUN_ID, "silver", "dim_venue") as step:
    venues = explode_latest("venues", S.VENUES, "venues").select(
        F.col("x.id").alias("venue_id"),
        F.col("x.name").alias("venue_name"),
        F.col("x.location.city").alias("city"),
        F.coalesce(F.col("x.location.stateAbbrev"), F.col("x.location.state")).alias("state"),
        F.col("x.location.country").alias("country"),
        F.col("x.location.defaultCoordinates.latitude").alias("latitude"),
        F.col("x.location.defaultCoordinates.longitude").alias("longitude"),
        F.col("x.location.elevation").alias("elevation_ft"),
        F.col("x.active").alias("is_active"),
        "extracted_at",
    )
    dim_venue = keep_latest(venues, ["venue_id"], ["extracted_at"]) \
        .withColumn("updated_at", F.current_timestamp()).drop("extracted_at")
    step.records = P.merge_upsert(spark, dim_venue, C.table("silver", "dim_venue"), ["venue_id"],
        comment="Silver: ballparks where in-scope games were played")

## dim_player
Height arrives as text (`6' 1"`) and is converted to inches; `is_dominican` drives the
Dominican players analysis in Gold.

In [ ]:
with P.log_step(spark, RUN_ID, "silver", "dim_player") as step:
    people = explode_latest("people", S.PEOPLE, "people")
    feet = F.regexp_extract("x.height", r"^(\d+)'", 1).cast("int")
    inches = F.regexp_extract("x.height", r"(\d+)\"", 1).cast("int")
    dim_player = keep_latest(people.select(
        F.col("x.id").alias("player_id"),
        F.col("x.fullName").alias("player_name"),
        F.col("x.firstName").alias("first_name"),
        F.col("x.lastName").alias("last_name"),
        F.to_date("x.birthDate").alias("birth_date"),
        F.col("x.birthCity").alias("birth_city"),
        F.col("x.birthStateProvince").alias("birth_state_province"),
        F.col("x.birthCountry").alias("birth_country"),
        (F.col("x.birthCountry") == C.DOMINICAN_COUNTRY).alias("is_dominican"),
        (feet * 12 + F.coalesce(inches, F.lit(0))).alias("height_in"),
        F.col("x.weight").alias("weight_lb"),
        F.col("x.primaryPosition.abbreviation").alias("primary_position"),
        F.col("x.primaryPosition.name").alias("primary_position_name"),
        F.col("x.primaryPosition.type").alias("position_type"),
        F.col("x.batSide.code").alias("bats"),
        F.col("x.pitchHand.code").alias("throws"),
        F.to_date("x.mlbDebutDate").alias("mlb_debut_date"),
        F.col("x.active").alias("is_active"),
        F.col("x.currentTeam.id").alias("current_team_id"),
        "extracted_at",
    ), ["player_id"], ["extracted_at"]).withColumn("updated_at", F.current_timestamp()).drop("extracted_at")
    step.records = P.merge_upsert(spark, dim_player, C.table("silver", "dim_player"), ["player_id"],
        comment="Silver: player profiles (birth country, position, handedness, debut) for every player in a boxscore")

## official_standings (reconciliation reference)

In [ ]:
with P.log_step(spark, RUN_ID, "silver", "official_standings") as step:
    recs = (P.latest_payloads(spark, "standings")
            .select(F.explode(F.from_json("payload", S.STANDINGS).records).alias("r"), "extracted_at")
            .select("r.division.id", F.explode("r.teamRecords").alias("t"), "extracted_at"))
    standings = keep_latest(recs.select(
        F.col("t.season").cast("int").alias("season"),
        F.col("t.team.id").alias("team_id"),
        F.col("id").alias("division_id"),
        F.col("t.gamesPlayed").alias("games_played"),
        F.col("t.wins").alias("wins"),
        F.col("t.losses").alias("losses"),
        F.col("t.runsScored").alias("runs_scored"),
        F.col("t.runsAllowed").alias("runs_allowed"),
        F.col("t.runDifferential").alias("run_differential"),
        F.col("t.winningPercentage").cast("double").alias("win_pct"),
        F.col("t.divisionRank").cast("int").alias("division_rank"),
        F.col("t.leagueRank").cast("int").alias("league_rank"),
        F.col("t.gamesBack").alias("games_back"),
        F.col("t.wildCardGamesBack").alias("wild_card_games_back"),
        F.col("t.clinchIndicator").alias("clinch_indicator"),
        F.col("t.streak.streakCode").alias("streak"),
        F.col("extracted_at").alias("as_of"),
    ), ["season", "team_id"], ["as_of"]).withColumn("updated_at", F.current_timestamp())
    step.records = P.merge_upsert(spark, standings, C.table("silver", "official_standings"), ["season", "team_id"],
        comment="Silver: official MLB regular-season standings, used to reconcile the W-L and runs computed from games")

In [ ]:
for t in ["dim_team", "dim_venue", "dim_player", "official_standings"]:
    print(f"{t:<20} {spark.table(C.table('silver', t)).count():>6} rows")
display(spark.sql(f"""SELECT birth_country, COUNT(*) AS players FROM {C.table('silver', 'dim_player')}
                      GROUP BY birth_country ORDER BY players DESC LIMIT 10"""))